# Data Mining Final Project: Computer Science Journal Finder

This notebook implements the required journal finder software and topic clustering workflow for the provided computer science publication database.

## 1. Load Libraries and Project Modules

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_dataset, summarize_dataset
from src.modeling import cluster_topics, train_vectorizer
from src.recommender import format_recommendations, recommend_journals

DB_PATH = PROJECT_ROOT / 'CompSciencePub.sqlite'
DB_PATH

WindowsPath('C:/Users/umut3/Downloads/DATA_MINING/CompSciencePub.sqlite')

## 2. Load and Inspect Dataset

In [2]:
df = load_dataset(DB_PATH)
summarize_dataset(df)

{'articles': 23061, 'journals': 455, 'subjects': 80}

In [3]:
df[['record_id', 'title', 'journal', 'subjects', 'training_text']].head()

,record_id,title,journal,subjects,training_text
0,88652,An updated survey of GA-based multiobjective o...,ACM COMPUTING SURVEYS,"Computer Science, Theory & Methods, Computer S...",an updated survey of ga-based multiobjective o...
1,88653,The state of the art in distributed query proc...,ACM COMPUTING SURVEYS,"Computer Science, Theory & Methods, Computer S...",the state of the art in distributed query proc...
2,88654,Logical models of argument,ACM COMPUTING SURVEYS,"Computer Science, Theory & Methods, Computer S...",logical models of argument logical models of a...
3,88655,Information retrieval on the Web,ACM COMPUTING SURVEYS,"Computer Science, Theory & Methods, Computer S...",information retrieval on the web in this paper...
4,88656,A guided tour to approximate string matching,ACM COMPUTING SURVEYS,"Computer Science, Theory & Methods, Computer S...",a guided tour to approximate string matching w...


## 3. Train TF-IDF Representation

In [4]:
vectorizer, matrix = train_vectorizer(df['training_text'])
matrix.shape

(23061, 40000)

## 4. Recommend Top-5 Journals for a New Abstract

In [5]:
sample_abstract = """
This paper proposes a deep learning method for detecting software defects from source code metrics and commit history.
The approach combines neural text representations with supervised classification to identify risky modules in large software projects.
Experiments evaluate precision, recall, and generalization across multiple open source repositories.
"""

recommendations = recommend_journals(sample_abstract, df, vectorizer, matrix, top_n=5)
format_recommendations(recommendations)

,rank,journal,score,matched_articles,example_title,subjects
0,1,IEEE SOFTWARE,0.1638,1,The many meanings of open source,"Computer Science, Software Engineering, Comput..."
1,2,JOURNAL OF SYSTEMS AND SOFTWARE,0.1464,2,Effective fault prediction model developed usi...,"Computer Science, Software Engineering, Comput..."
2,3,IEEE TRANSACTIONS ON SOFTWARE ENGINEERING,0.1439,7,Empirical validation of object-oriented metric...,"Computer Science, Software Engineering, Engine..."
3,4,AUTOMATED SOFTWARE ENGINEERING,0.1365,8,Maintainability defects detection and correcti...,"Computer Science, Software Engineering, Comput..."
4,5,EMPIRICAL SOFTWARE ENGINEERING,0.1349,10,Curating GitHub for engineered software projects,"Computer Science, Software Engineering, Comput..."


## 5. Generate Topic Clusters

In [6]:
clusters = cluster_topics(df, n_clusters=8)
clusters

,cluster,article_count,top_terms,dominant_subjects,sample_journals
0,7,5663,"software, systems, information, engineering, s...","Computer Science, Software Engineering, Inform...",COMPUTER APPLICATIONS IN ENGINEERING EDUCATION...
1,1,4512,"artificial, intelligence, artificial intellige...","Computer Science, Artificial Intelligence, Eng...","COMPUTER SPEECH AND LANGUAGE, MEDICAL IMAGE AN..."
2,4,4043,"mathematics, theory, methods, theory methods, ...","Computer Science, Mathematics, Theory & Method...","INFORMATION AND COMPUTATION, CMES-COMPUTER MOD..."
3,0,3088,"architecture, hardware, cloud, computing, hard...","Computer Science, Hardware & Architecture, The...","INTEGRATION-THE VLSI JOURNAL, ACM TRANSACTIONS..."
4,3,2890,"telecommunications, networks, wireless, networ...","Telecommunications, Computer Science, Engineer...","IEEE COMMUNICATIONS MAGAZINE, IEEE JOURNAL ON ..."
5,2,1371,"optimization, algorithm, search, problems, alg...","Computer Science, Operations Research & Manage...","OPTIMIZATION METHODS & SOFTWARE, INFORMS JOURN..."
6,6,791,"biology, computational biology, mathematical c...","Mathematical & Computational Biology, Computer...","COMPUTERS IN BIOLOGY AND MEDICINE, JOURNAL OF ..."
7,5,703,"science library, library science, information ...","Computer Science, Information Science & Librar...","JOURNAL OF MANAGEMENT INFORMATION SYSTEMS, ONL..."


## 6. Notes for Report

- The project uses article abstracts, titles, keywords, keyword plus terms, and subject categories.
- TF-IDF and cosine similarity produce the journal ranking.
- KMeans produces subject/topic clusters.
- The Streamlit app in `app.py` is the interactive journal finder software.